## Simulating traffic around a lock
In this notebook, we simulate a lock on a network which randomly generated vessels have to pass. We add a pre-coded complex lock object on the graph. Vessels are locked together if they can fit inside the lock, and arrive within the clustering time window.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.utils import create_object
from opentnsim.utils import generate_vessels_from_distribution
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


#### 0. Create environment

In [2]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-25000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(25000,0)))

# add edges
graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-25000, 0),Point(-5000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(25000, 0),Point(5000, 0)])), weight=1, length_m=25000-5000)

# add graph to environment
env.graph = graph

In [4]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure

In [5]:
lock_chamber_I = IsLockChamber(env=env,
                               lock_length = 200,
                               lock_width = 20,
                               lock_depth = 6,
                               name='Lock chamber I',
                               gate_open = '0',
                               edge = ('0','1'),
                               geometry_m = Polygon([Point(-100, -10),Point(-100, 10),Point(100, 10),Point(100, -10)]))

lock_chamber_II = IsLockChamber(env=env,
                                lock_length = 200,
                                lock_width = 20,
                                lock_depth = 6,
                                name='Lock chamber II',
                                gate_open = '0',
                                edge = ('0','1'),
                                geometry_m = Polygon([Point(-100, -10),Point(-100, 10),Point(100, 10),Point(100, -10)]))

In [6]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   distance_from_edge_start = 0)

In [7]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber_I, lock_chamber_II],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents

In [8]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
)

In [9]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

In [10]:
upstream_vessels = generate_vessels_from_distribution(env=env,
                                                      VesselClass = Vessel,
                                                      vessel_parameters = {'v':4, 'L':135, 'B':17, 'T':5, 'type':'tanker'},
                                                      mean_arrival_rate=30.,
                                                      number_of_vessels=2,
                                                      start_node = '-1',
                                                      end_node = '+1',
                                                      seed = 123)

downstream_vessels = generate_vessels_from_distribution(env=env,
                                                        VesselClass = Vessel,
                                                        vessel_parameters = {'v':4, 'L':135, 'B':17, 'T':5, 'type':'tanker'},
                                                        mean_arrival_rate=30.,
                                                        number_of_vessels=2,
                                                        start_node = '+1',
                                                        end_node = '-1',
                                                        seed = 456)

vessels = upstream_vessels + downstream_vessels

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation

In [11]:
env.run()

#### 4. Inspect output

In [12]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber_I.logbook)

print("'{}' logbook data:".format(lock_chamber_I.name))  
print('')

display(lock_df)

'Lock chamber I' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-01 01:38:04.354414,{},0
1,Lock gate closing stop,2025-01-01 01:43:04.354414,{},0
2,Lock levelling start,2025-01-01 01:43:04.354414,{},0
3,Lock levelling stop,2025-01-01 01:53:04.354414,{},1
4,Lock gate opening start,2025-01-01 01:53:04.354414,{},1
5,Lock gate opening stop,2025-01-01 01:58:04.354414,{},1
6,Lock gate closing start,2025-01-01 02:30:07.714638,{},1
7,Lock gate closing stop,2025-01-01 02:35:07.714638,{},1
8,Lock levelling start,2025-01-01 02:35:07.714638,{},1
9,Lock levelling stop,2025-01-01 02:45:07.714638,{},0


In [13]:
# load the logbook data into a dataframe
lock_df = pd.DataFrame.from_dict(lock_chamber_II.logbook)

print("'{}' logbook data:".format(lock_chamber_II.name))  
print('')

display(lock_df)

'Lock chamber II' logbook data:



,Message,Timestamp,Value,Geometry
0,Lock gate closing start,2025-01-01 02:03:48.330189,{},0
1,Lock gate closing stop,2025-01-01 02:08:48.330189,{},0
2,Lock levelling start,2025-01-01 02:08:48.330189,{},0
3,Lock levelling stop,2025-01-01 02:18:48.330189,{},1
4,Lock gate opening start,2025-01-01 02:18:48.330189,{},1
5,Lock gate opening stop,2025-01-01 02:23:48.330189,{},1
6,Lock gate closing start,2025-01-01 02:56:27.099087,{},1
7,Lock gate closing stop,2025-01-01 03:01:27.099087,{},1
8,Lock levelling start,2025-01-01 03:01:27.099087,{},1
9,Lock levelling stop,2025-01-01 03:11:27.099087,{},0


#### Gantt chart of event table

In [14]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber_I])
fig = generate_vessel_gantt_chart(df_eventtable)

#### Time-distance diagram of vessels passing the lock and planning info

In [15]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber_I.plot(xlimmin = -6050, 
                          xlimmax = 6050,
                          ylimmin = pd.Timestamp('2025-01-01 01:00:00'),
                          ylimmax = pd.Timestamp('2025-01-01 09:00:00'),
                          method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

In [16]:
# We can plot the time-distance diagram
fig = lock_chamber_II.plot(xlimmin = -6050, 
                           xlimmax = 6050,
                           ylimmin = pd.Timestamp('2025-01-01 01:00:00'),
                           ylimmax = pd.Timestamp('2025-01-01 09:00:00'),
                           method='Plotly')

fig.update_layout(height=cm_to_pixels(20))

#### Vessel delays: individual delays and overall average

In [17]:
delays = []
for vessel in vessels:
    vessel_df = pd.DataFrame(vessel.logbook)
    waiting_start = vessel_df[vessel_df.Message.str.match(r"^Waiting .* start$", na=False)]
    waiting_stop = vessel_df[vessel_df.Message.str.match(r"^Waiting .* stop$", na=False)]
    if not waiting_stop.empty:
        delay = (waiting_stop.Timestamp.values-waiting_start.Timestamp.values).sum()
    else:
        delay = pd.Timedelta(seconds=0)
    delays.append(delay)

In [18]:
print(f"The average vessel delay is {np.round(np.mean(delays)/np.timedelta64(1,'s')/60,1)} minutes")

The average vessel delay is 57.6 minutes
